[Sutton & Barto RL Book]: http://incompleteideas.net/book/RLbook2020.pdf

## Introduction
This section contains methods from Chapter 10 in [Sutton & Barto RL Book].

### Linear Methods
Approximate functions $\hat{v}(\cdot, \mathbf{w})$ or $\hat{q}(\cdot, \mathbf{w})$,
are a linear function of the weight vector $\mathbf{w}$.
Linear methods approximate the value functions by the inner product
between $\mathbf{w}$ and $\mathbf{x}(s)$ - for the state-value
function $\hat{v}(\cdot, \mathbf{w})$; or
$\mathbf{x}(s, a)$ for $\hat{q}(\cdot, \mathbf{w})$ for the state-action value function, where $\mathbf{x}(s)$ or $\mathbf{x}(s, a)$ are feature vectors, encoding the state
or state-action pairs into a feature space.

The value functions are defined as:
* $\hat{v}(s, \mathbf{w}) \overset{\cdot}{=} \mathbf{w}^\top \mathbf{x}(s) \overset{\cdot}{=} \sum_{i=1}^{d} w_i x_i(s)$
* $\hat{q}(s, a, \mathbf{w}) \overset{\cdot}{=} \mathbf{w}^\top \mathbf{x}(s, a) \overset{\cdot}{=} \sum_{i=1}^{d} w_i x_i(s, a)$

The gradient of the approximate value functions with respect to $\mathbf{w}$
in this case are:
* $\nabla \hat{v}(s, \mathbf{w}) = \mathbf{x}(s)$
* $\nabla \hat{q}(s, a, \mathbf{w}) = \mathbf{x}(s, a)$

### Prediction
For prediction we are interested in the state value estimation.
In the tabular case a continuous measure of prediction quality was not
necessary because the learned value function could come to equal the true
_value function_ exactly. Additionally, the learned  values at each state
were decoupled, an update at one state affected no other. But with approximation
an update at one state affects many others, and it is not possible to get the
values of all states exactly correct. Moreover, we assume that we have many
more states than we have weights, so making a few of the state estimations
correct, invariably makes the other states' estimations less so.

### Prediction Objective
Because we cannot get an accurate estimation for all states, we must emphasize
which states we care more about, by specifying a _state_ distribution
$\mu(s) \ge 0$ such that $\sum_{s}\mu(s) = 1$ representing how much we care
about the error at each state. We define the _error_ at each state as the
squared difference between the _approximate_ value $\hat{v}(s, \mathbf{w})$ and
the _true_ value $v(s)$. Weighing it by the state distribution $\mu(s)$ we get
an objective function called _mean square value error_ $\overline{VE}$ defined
as:

$\overline{VE}(\mathbf{w}) \overset{\cdot}{=}\sum_{s \in \mathcal{S}}\mu(s)[v(s) - \hat{v}(s, \mathbf{w})]^2$

$\mu(s)$ is usually chosen to be the fraction of time spent in $s$. Under the
on-policy training this is called the _on-policy_ distribution. In continuing
tasks, this distribtution is the stationary distribution under $\pi$.

An ideal goal in terms of $\overline{VE}$ would be to find the _global optimum_
$\mathbf{w}^*$ such that $\overline{VE}(\mathbf{w}^*) \le \overline{VE}(\mathbf{w})$
for all possible $\mathbf{w}$. This is rarely achieved for complex functions,
so we typically settle on _local optimum_, a weight vector $\mathbf{w}^*$
for which $\overline{VE}(\mathbf{w}^*) \le \overline{VE}(\mathbf{w})$ in a
neighborhood around $\mathbf{w}^*$.

### Stochastic Gradient
We assume that states appear in examples with the same distribution $\mu$ over
which we minimize $\overline{VE}$.  If we assume that on each step, we observe
a new example $S_t \rightarrow v_{\pi}(S_t)$, where $v_{\pi}(S_t)$ denotes
the _true_ value  of state $S_t$ under $\pi$, we converge to a local optimum
on the observed  examples using Stochastic Gradient Descend (SGD), by adjusting
the weight by  a small amount in the direction that would mostly reduce the
error in that example as follows:

$$
\mathbf{w}_{t + 1} = \mathbf{w}_t - \frac{1}{2}\alpha\nabla_\mathbf{w}[v_{\pi}(S_t) - \hat{v}(S_t, \mathbf{w}_t)]^2
= \mathbf{w}_t + \alpha[v_{\pi}(S_t) - \hat{v}(S_t, \mathbf{w}_t)]\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w}_t)
$$

Convergence results for SGD methods assume that $\alpha$ decreases over time
in order for the stochastic approximation to converge to a _local optimum_.

However, we don't usually have access to the true state value $v_{\pi}(S_t)$, and
instead we use a random approximation of it $U_t$. This can be a noise corrupted
approximation of $v_{\pi}(S_t)$ or a bootstrapped approximation. In these cases
we cannot perform an exact update - since  $v_{\pi}(S_t)$ is unknown - but
we can approximate it by substituting it in the SDG formulation as follows:

$$
\mathbf{w}_{t + 1} = \mathbf{w}_t + \alpha[U_t - \hat{v}(S_t, \mathbf{w}_t)]\nabla_\mathbf{w}\hat{v}(S_t, \mathbf{w}_t)
$$

#### Unbiased Target
$U_t$ can be an unbiased estimate of $v_{\pi}(S_t)$, i.e. $\mathbb{E}[U_t | S_t = s] = v_{\pi}(s)$
for each $t$, then $\mathbf{w}_t$ is guaranteed to converge to a local optimum.
Under the Monte-Carlo setting, the policy generates the sequence of examples.
Since the true value of the return is the expectation of the return
following policy $\pi$, then $U_t = G_t$ is definitionally an
unbiased estimate of $v_{\pi}(S_t)$.

#### Semi-Gradient
However, when we use bootstrapping, we do not obtain the same guarantees for
the estimation of $v_{\pi}(S_t)$. For the n-step return:
$G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+2} + ... + \gamma^{n}\hat{v}(S_{t+n}, \mathbf{w}_t)$
$G_{t:t+n}$ as well as $\hat{v}(*, \mathbf{w_t})$ both depend on the value
of $\mathbf{w}_t$. Therefore, the gradient formulation above does  not produce
a _true_ gradient. It changes the weights based on the estimate but ignores
the effects of it in the target. Therefore, they only include part of the
gradient and are called _semi-gradient methods_. Although semi-gradient methods
do not have robust convergence guarantees, they  typically enable significantly
faster learning, can be used for online learning,  and computational advantages.
For TD(0), the target becomes $U_t \overset{\cdot}{=} R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}_t)$


### Episodic Control
For control, we switch our focus toward action (or state-action) value function approximation $\hat{q} \approx q_{\pi}$ parametrized by $\mathbf{w}$.
Similar to the prediction case $S_t \rightarrow U_t$, our target for the control case becomes $S_t, A_t \rightarrow U_t$.
The update target $U_t$ can be any approximation of $q_{\pi}(St, At)$, including the backed-up values of the full Monte Carlo return ($G_t$) or any of the $n$-step Sarsa ($G_{t:t+n}$)

#### One-Step Semi-Gradient
Carrying on the analogy from the prediction case, the gradient descend update for control is

$
\mathbf{w}_{t + 1}  \overset{\cdot}{=} \mathbf{w}_t + \alpha[U_t - \hat{q}(S_t, A_t, \mathbf{w}_t)]\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w}_t)
$

Analogous to the tabular case, the target for the approximate function control can be formulated as follows for the different Sarsa flavors

* Sarsa:
$U_t = R_{t+1} + \gamma \hat{q}(S_{t + 1}, A_{t + 1}, \mathbf{w}_t)$
* Expected Sarsa:
$U_t = R_{t+1} + \gamma \sum_{a}\pi(a|S_{t+1})\hat{q}(S_{t + 1}, a, \mathbf{w}_t)$
* Sarsa-max (aka, Q-Learning):
$U_t = R_{t+1} + \gamma \max_{a}\hat{q}(S_{t + 1}, a, \mathbf{w}_t)$

To form control methods, we need to couple such action-value prediction methods with techniques for _policy improvement_ and _action selection_.
For smaller set actions the tabular methods are easily extended to incorporate the function approximation methods in their algorithms.

##### Action Selection
For each possible action $a$ available in the next state $S_{t+1}$, we can compute $\hat{q}(S_{t+1}, a, \mathbf{w}_t)$ and then find the greedy action $A^*_{t + 1} = \arg\max_a(\hat{q}(S_{t+1}, a, \mathbf{w}_t))$

##### Policy Improvement
For the on-policy case, policy improvement is achieved by changing the estimation policy to a soft approximation of the greedy policy such as the _$\varepsilon$-greedy_ policy


#### $n$-Step Semi-Gradient

We can obtain an n-step version of episodic semi-gradient Sarsa by using an $n$-step return as the update target $U_t = G_{t:t+n}$ in the semi-gradient Sarsa (and related flavors) update equations.

* Sarsa: $G_{t:t+n} \overset{\cdot}{=} R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \hat{q}(S_{t+n}, A_{t+n}, \mathbf{w}_{t+n-1}), \ t + n < T$
* Expected Sarsa: $G_{t:t+n} \overset{\cdot}{=} R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n{\sum_{a}{\pi(a|S_{t+n})\hat{q}(S_{t+n}, a, \mathbf{w}_{t+n-1})}}, \ t + n < T$
* SarsaMax / Q-Learning: $G_{t:t+n} \overset{\cdot}{=} R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n{\max_{a}\hat{q}(S_{t+n}, a, \mathbf{w}_{t+n-1})}, \ t + n < T$

where $G_{t:t+n} \overset{\cdot}{=} G_t$ for $t + n \ge T$

where the gradient update - just like in the one step-case - becomes:

$
\mathbf{w}_{t + n}  \overset{\cdot}{=} \mathbf{w}_{t + n - 1} + \alpha[G_{t:t+n} - \hat{q}(S_t, A_t, \mathbf{w}_{t + n - 1})]\nabla_\mathbf{w}\hat{q}(S_t, A_t, \mathbf{w}_{t + n - 1})
$


#### Continuous Control: Average Reward
This setting applies to continuing problems, problems for which the interaction between agent and environment goes on and on forever, without termination or start states.
Here we have no discounting, the agent cares just as much about delayed rewards as it does about immediate reward.

The quality of a policy $\pi$ is defined as the average rate of reward, or simply _average reward_, while following that policy, which we denote as $r(\pi)$

$$
\begin{align}
r(\pi) &\overset{\cdot}{=} \lim_{h \to \infty} \frac{1}{h} \sum_{t=1}^{h} \mathbb{E}[R_t \mid S_0, A_{0:t-1} \sim \pi] &\quad \text{(1)} \\
       &= \lim_{t \to \infty} \mathbb{E}[R_t \mid S_0, A_{0:t-1} \sim \pi] &\quad \text{(2)} \\
       &= \sum_s \mu_\pi(s) \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a) r &\quad \text{(3)}
\end{align}
$$

$(1)$ defines the average reward as the average $\dfrac{1}{h}$ accross the temporal rollout of the process of the expectations of the RV's $R_t$ - conditioned on the initial state $S_0$ and ensuing
actions $A_0, A_1, ...$ following policy $\pi$. If we assume that the steady state distribution $\mu_{\pi}(s) = \lim_{t \rightarrow \infty} P\{ S_{t} = s| A_{0:t} \sim \pi \}$ exists, and is
independent of the initial state $S_0$, i.e. the MDP is _ergodic_, we can recover the average reward $(1)$ over the temporal rollout $(2)$ which is the sample expectation of $(3)$ over SSD $\mu_{\pi}$.

We order policies according to their average reward per time step according to their average reward or reward rate $r(\pi)$.
In particular, we consider policies that attain the maximal value of $r(\pi)$ to be optimal. In the average-reward setting, returns are defined in terms of differences
between rewards and the average reward:

$$
    G_{t} \overset{\cdot}{=} R_{t+1} - r(\pi) + R_{t+2} - r(\pi) + ...
$$

which is called the __differential__ return and the corresponding value functions are called differential value functions. Differential value functions are defined in terms
of the new return just as conventional value functions were defined in terms of the discounted return; thus we will use the same notation as in the episodic case.

These differential equations also have Bellman Equations:

$$
\begin{aligned}
v_\pi(s) &= \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a) \left[ r - r(\pi) + v_\pi(s') \right], \\
q_\pi(s, a) &= \sum_{s', r} p(s', r \mid s, a) \left[ r - r(\pi) + \sum_{a'} \pi(a' \mid s') q_\pi(s', a') \right], \\
v_*(s) &= \max_a \sum_{s', r} p(s', r \mid s, a) \left[ r - \max_\pi r(\pi) + v_*(s') \right], \quad \text{and} \\
q_*(s, a) &= \sum_{s', r} p(s', r \mid s, a) \left[ r - \max_\pi r(\pi) + \max_{a'} q_*(s', a') \right]
\end{aligned}
$$




#### One-Step Differential Semi-Gradient

The temporal difference (TD) errors defined as:
* For prediction
$$ \delta_t \overset{\cdot}{=} R_{t+1} - \bar{R}_t + \hat{v}(S_{t+1}, \mathbf{w}_t) - \hat{v}(S_t, \mathbf{w}_t) $$

* For control
$$ \delta_t \overset{\cdot}{=} R_{t+1} - \bar{R}_t + \hat{q}(S_{t+1}, A_{t+1}, \mathbf{w}_t) - \hat{q}(S_t, A_t, \mathbf{w}_t) $$

where $\bar{R}_t$ is also an estimate of $r(\pi)$ at $t$.

Accordingly, the semi-gradient update then becomes:
* For prediction
$$ \mathbf{w}_{t+1} \overset{\cdot}{=} \mathbf{w}_t + \alpha \delta_t \nabla \hat{v}(S_t, \mathbf{w}_t) $$

* For control
$$ \mathbf{w}_{t+1} \overset{\cdot}{=} \mathbf{w}_t + \alpha \delta_t \nabla \hat{q}(S_t, A_t, \mathbf{w}_t) $$

#### $n$-Step Differential Semi-Gradient
For the $n$-Step case, same as we did under the episodic case, the (control) Sarsa target is:

$$ G_{t:t+n} \overset{\cdot}{=} R_{t+1} - \bar{R}_{t} + R_{t+2} - \bar{R}_{t+1} + \cdots + R_{t+n} - \bar{R}_{t+n-1} + \hat{q}(S_{t+n}, A_{t+n}, \mathbf{w}_{t+n-1}) $$

and the gradient update is:
$$ \delta_t \overset{\cdot}{=} G_{t:t+n} - \hat{q}(S_t, A_t, \mathbf{w}) $$

For Expected Sarsa we can use the following target:
$$ G_{t:t+n} \overset{\cdot}{=} R_{t+1} - \bar{R}_{t} + R_{t+2} - \bar{R}_{t+1} + \cdots + R_{t+n} - \bar{R}_{t+n-1} + \sum_{a} \pi(a | S_{t+n}) \hat{q}(S_{t+n}, a, \mathbf{w}_{t+n-1}) $$

And for Sarsa-Max (Q-Learning) we can use the following target:
$$ G_{t:t+n} \overset{\cdot}{=} R_{t+1} - \bar{R}_{t} + R_{t+2} - \bar{R}_{t+1} + \cdots + R_{t+n} - \bar{R}_{t+n-1} + \max_{a} \hat{q}(S_{t+n}, a, \mathbf{w}_{t+n-1}) $$

Note that for Q-Learning $\bar{R}_{t}$ is supposed to approximate $\max_{\pi}r(\pi)$. But since this is an approximation of the average return of the optimal $q_{*}$, the sample averages are also an
approximation of $\max_{\pi}r(\pi)$. So, we track the average reward just like in the Sarsa case